In [1]:
import vitaldb
import pandas as pd
import numpy as np
from scipy.stats import linregress
from tqdm.auto import tqdm

TRACKS = [
    "Solar8000/HR",
    "Solar8000/PLETH_SPO2",
    "Solar8000/ART_MBP"
]

INPUT_MINUTES = 15
PREDICTION_MINUTES = 30
STRIDE_MINUTES = 5

INPUT_SECONDS = INPUT_MINUTES * 60
PREDICTION_SECONDS = PREDICTION_MINUTES * 60
STRIDE_SECONDS = STRIDE_MINUTES * 60


def slope(series):
    series = series.dropna()

    if len(series) < 2:
        return np.nan

    return linregress(
        np.arange(len(series)),
        series.values
    ).slope


def generate_windows(df):

    windows = []

    total_len = len(df)

    start = 0

    while (
        start
        + INPUT_SECONDS
        + PREDICTION_SECONDS
        <= total_len
    ):

        input_df = df.iloc[
            start :
            start + INPUT_SECONDS
        ]

        prediction_df = df.iloc[
            start + INPUT_SECONDS :
            start + INPUT_SECONDS + PREDICTION_SECONDS
        ]

        windows.append(
            (input_df, prediction_df)
        )

        start += STRIDE_SECONDS

    return windows

c:\Users\Raj Jaiswal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def extract_trend_features(input_df):

    hr = input_df["Solar8000/HR"]
    spo2 = input_df["Solar8000/PLETH_SPO2"]
    map_ = input_df["Solar8000/ART_MBP"]

    return {

        # HR
        "hr_mean": hr.mean(),
        "hr_std": hr.std(),
        "hr_slope": slope(hr),

        # SpO2
        "spo2_mean": spo2.mean(),
        "spo2_std": spo2.std(),
        "spo2_slope": slope(spo2),

        # MAP
        "map_mean": map_.mean(),
        "map_std": map_.std(),
        "map_slope": slope(map_)
    }

In [4]:
def create_labels(prediction_df):

    hr = prediction_df["Solar8000/HR"]
    spo2 = prediction_df["Solar8000/PLETH_SPO2"]
    map_ = prediction_df["Solar8000/ART_MBP"]

    return {

        "tachycardia":
            int(hr.max() > 110),

        "hypotension":
            int(map_.min() < 65),

        "hypoxia":
            int(spo2.min() < 90)
    }

In [5]:
def process_case(caseid):

    try:

        arr = vitaldb.load_case(
            caseid,
            TRACKS,
            interval=1
        )

        df = pd.DataFrame(
            arr,
            columns=TRACKS
        )

        # =====================================
        # Clean invalid values
        # =====================================

        df["Solar8000/HR"] = (
            df["Solar8000/HR"]
            .mask(
                df["Solar8000/HR"] <= 0
            )
        )

        df["Solar8000/PLETH_SPO2"] = (
            df["Solar8000/PLETH_SPO2"]
            .mask(
                df["Solar8000/PLETH_SPO2"] <= 0
            )
        )

        df["Solar8000/ART_MBP"] = (
            df["Solar8000/ART_MBP"]
            .mask(
                df["Solar8000/ART_MBP"] < 30
            )
        )

        # =====================================
        # Skip useless cases
        # =====================================

        if (
            df["Solar8000/ART_MBP"]
            .notna()
            .sum()
            < 900
        ):
            return []

        windows = generate_windows(df)

        rows = []

        for idx, (
            input_df,
            prediction_df
        ) in enumerate(windows):

            hr = input_df["Solar8000/HR"]
            spo2 = input_df["Solar8000/PLETH_SPO2"]
            map_ = input_df["Solar8000/ART_MBP"]

            # =====================================
            # Quality filter
            # =====================================

            if (
                hr.notna().sum() < 300
                or spo2.notna().sum() < 300
                or map_.notna().sum() < 300
            ):
                continue

            # =====================================
            # VitalML Stability Filter
            # Patient must be stable during
            # assessment window
            # =====================================

            stable = (
                hr.max() <= 110
                and
                spo2.min() >= 90
                and
                map_.min() >= 65
            )

            if not stable:
                continue

            # =====================================
            # Features
            # =====================================

            feats = extract_trend_features(
                input_df
            )

            # =====================================
            # Future labels
            # =====================================

            labels = create_labels(
                prediction_df
            )

            rows.append({

                "caseid": caseid,
                "window_id": idx,

                **feats,

                **labels
            })

        return rows

    except Exception as e:

        print(
            f"Case {caseid} failed: {e}"
        )

        return []

In [9]:
import pandas as pd

df = pd.read_csv("window_features_stable.csv")

caseids = sorted(df["caseid"].unique())

print(len(caseids))

2986


In [ ]:
all_rows = []

for caseid in tqdm(caseids):

    rows = process_case(caseid)

    all_rows.extend(rows)

feature_df = pd.DataFrame(all_rows)

feature_df.to_csv(
    "window_features_stable.csv",
    index=False
)

print(feature_df.shape)

print("Tachycardia :", feature_df["tachycardia"].mean())
print("Hypotension :", feature_df["hypotension"].mean())
print("Hypoxia     :", feature_df["hypoxia"].mean())

In [11]:
import pandas as pd

df = pd.read_csv("window_features_stable.csv")

print(df.shape)
print(df.head())

print(df.isna().sum())

(52638, 14)
   caseid  window_id    hr_mean     hr_std  hr_slope  spo2_mean  spo2_std  \
0       1          9  79.388885  15.223864 -0.114357  98.682220  1.149763   
1       1         10  68.786670   7.751934 -0.041341  97.428890  1.122978   
2       1         18  67.766670   4.552710 -0.016620  98.726670  0.628535   
3       1         19  66.873340   2.575272  0.012670  98.924446  0.264580   
4       1         20  68.904440   2.728588  0.019920  98.942220  0.233582   

   spo2_slope    map_mean    map_std  map_slope  tachycardia  hypotension  \
0   -0.008501  101.566666  14.593719  -0.109927            0            1   
1   -0.008259   86.917780  12.488209  -0.091793            0            1   
2    0.001790   80.773330   6.094446  -0.028771            0            0   
3    0.000096   79.148890   3.916450  -0.003311            0            0   
4    0.000513   80.724440   4.745137   0.035380            0            0   

   hypoxia  
0        0  
1        0  
2        0  
3        0

In [12]:
import pandas as pd

# Load
df = pd.read_csv("window_features_stable.csv")

print("Original shape:", df.shape)

# =====================================
# Remove rows with missing features
# =====================================

df = df.dropna(
    subset=[
        "hr_mean",
        "hr_std",
        "hr_slope",
        "spo2_mean",
        "spo2_std",
        "spo2_slope",
        "map_mean",
        "map_std",
        "map_slope"
    ]
)

print("After NaN removal:", df.shape)

# =====================================
# Remove physiologically impossible values
# =====================================

df = df[
    (df["map_mean"] >= 30) &
    (df["map_mean"] <= 200) &

    (df["hr_mean"] >= 20) &
    (df["hr_mean"] <= 220) &

    (df["spo2_mean"] >= 50) &
    (df["spo2_mean"] <= 100)
]

print("After outlier removal:", df.shape)

# =====================================
# Check label balance
# =====================================

print("\nLabel rates:")
print("Tachycardia :", df["tachycardia"].mean())
print("Hypotension :", df["hypotension"].mean())
print("Hypoxia     :", df["hypoxia"].mean())

# =====================================
# Save cleaned dataset
# =====================================

df.to_csv(
    "window_features_clean.csv",
    index=False
)

print("\nSaved: window_features_clean.csv")

Original shape: (52638, 14)
After NaN removal: (52638, 14)
After outlier removal: (52583, 14)

Label rates:
Tachycardia : 0.17726261339216096
Hypotension : 0.4055112869178252
Hypoxia     : 0.07574691440199303

Saved: window_features_clean.csv


In [13]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("window_features_clean.csv")

FEATURES = [
    "hr_mean",
    "hr_std",
    "hr_slope",
    "spo2_mean",
    "spo2_std",
    "spo2_slope",
    "map_mean",
    "map_std",
    "map_slope"
]

groups = df["caseid"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=groups)
)

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Train:", train_df.shape)
print("Test :", test_df.shape)

print(
    "Train patients:",
    train_df["caseid"].nunique()
)

print(
    "Test patients:",
    test_df["caseid"].nunique()
)

Train: (42237, 14)
Test : (10346, 14)
Train patients: 2387
Test patients: 597


In [15]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

X_train = train_df[FEATURES]
y_train = train_df["hypotension"]

X_test = test_df[FEATURES]
y_test = test_df["hypotension"]

model_h = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model_h.fit(X_train, y_train)

pred_prob = model_h.predict_proba(X_test)[:,1]

auc = roc_auc_score(
    y_test,
    pred_prob
)

print("Hypotension AUROC:", auc)

[LightGBM] [Info] Number of positive: 17002, number of negative: 25235
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001104 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2289
[LightGBM] [Info] Number of data points in the train set: 42237, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.402538 -> initscore=-0.394901
[LightGBM] [Info] Start training from score -0.394901
Hypotension AUROC: 0.6157701315874129


In [18]:
import pandas as pd

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_h.feature_importances_
})

print(
    importance.sort_values(
        "importance",
        ascending=False
    )
)

      feature  importance
0     hr_mean        1388
7     map_std        1244
6    map_mean        1240
1      hr_std        1144
8   map_slope         993
2    hr_slope         935
3   spo2_mean         827
4    spo2_std         730
5  spo2_slope         499


In [19]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

X_train = train_df[FEATURES]
y_train = train_df["tachycardia"]

X_test = test_df[FEATURES]
y_test = test_df["tachycardia"]

model_t = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model_t.fit(X_train, y_train)

pred_prob = model_t.predict_proba(X_test)[:,1]

auc = roc_auc_score(
    y_test,
    pred_prob
)

print("Tachycardia AUROC:", auc)

[LightGBM] [Info] Number of positive: 7475, number of negative: 34762
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000327 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2289
[LightGBM] [Info] Number of data points in the train set: 42237, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.176978 -> initscore=-1.536961
[LightGBM] [Info] Start training from score -1.536961
Tachycardia AUROC: 0.6727488369128799


In [20]:
import pandas as pd

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_t.feature_importances_
})

print(
    importance.sort_values(
        "importance",
        ascending=False
    )
)

      feature  importance
0     hr_mean        1387
6    map_mean        1278
1      hr_std        1239
7     map_std        1182
2    hr_slope        1066
8   map_slope         890
3   spo2_mean         722
4    spo2_std         665
5  spo2_slope         571


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

X_train = train_df[FEATURES]
y_train = train_df["hypoxia"]

X_test = test_df[FEATURES]
y_test = test_df["hypoxia"]

model_hy= LGBMClassifier(
    n_estimators=550,
    learning_rate=0.005,
    num_leaves=42,
    random_state=42
)

model_hy.fit(X_train, y_train)

pred_prob = model_hy.predict_proba(X_test)[:,1] 

auc = roc_auc_score(
    y_test,
    pred_prob
)

print("Hypoxia AUROC:", auc)

[LightGBM] [Info] Number of positive: 3168, number of negative: 39069
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006515 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2289
[LightGBM] [Info] Number of data points in the train set: 42237, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.075005 -> initscore=-2.512229
[LightGBM] [Info] Start training from score -2.512229
Hypoxia AUROC: 0.6215692673503896
